In [ ]:
# Шаг 1: Установка необходимых библиотек
!pip install -q -U google-genai pydantic

In [ ]:
from pydantic import BaseModel, Field, field_validator
from typing import List, Optional

# Модель для конкретного просмотренного фильма
class WatchedMovie(BaseModel):
    title: str = Field(description="Название фильма или сериала")
    genre: str = Field(description="Основной жанр (например: Фантастика, Драма, Комедия)")
    rating: Optional[float] = Field(default=None, description="Оценка пользователя от 1.0 до 10.0")
    watch_time_minutes: int = Field(description="Время просмотра в минутах")
    completion_rate: float = Field(description="Процент досмотра от 0 до 100")

    @field_validator('watch_time_minutes')
    @classmethod
    def check_watch_time(cls, v: int) -> int:
        if v <= 0:
            raise ValueError('Время просмотра должно быть больше нуля')
        return v

    @field_validator('completion_rate')
    @classmethod
    def check_completion(cls, v: float) -> float:
        if not (0 <= v <= 100):
            raise ValueError('Процент досмотра должен быть в диапазоне от 0 до 100')
        return v

    @field_validator('rating')
    @classmethod
    def check_rating(cls, v: Optional[float]) -> Optional[float]:
        if v is not None and not (1.0 <= v <= 10.0):
            raise ValueError('Оценка должна быть от 1 до 10')
        return v

# Модель профиля и предпочтений пользователя
class UserProfile(BaseModel):
    user_id: str = Field(description="Идентификатор пользователя")
    watch_history: List[WatchedMovie] = Field(description="История просмотров")
    top_genres: List[str] = Field(description="Основные любимые жанры на основе истории")
    engagement_level: str = Field(description="Уровень вовлеченности: Высокий, Средний, Низкий")
    recommended_movies: List[str] = Field(description="Список из 3-5 рекомендованных к просмотру фильмов")

In [ ]:
from google import genai
from google.colab import userdata

# Подключение к Gemini API через секреты Colab
api_key = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=api_key)

print("Gemini API успешно подключен!")

Gemini API успешно подключен!


In [ ]:
from google.genai import types
import json

# Неструктурированный лог активности пользователя кинотеатра
user_log_text = """
Пользователь с кодом USR-9041 провел у нас вечер.
Сначала он включил научно-фантастический фильм 'Интерстеллар', смотрел 169 минут и досмотрел его полностью (100% досмотра), поставив оценку 9.5 из 10.
Затем запустил комедию 'Мальчишник в Вегасе', но выключил через 20 минут (процент досмотра всего 20%), оценку не оставил.
Позже посмотрел фантастику 'Начало' (продолжительность просмотра 148 минут, 100% досмотра), поставил 9.0 баллов.
"""

# Запрос к Gemini с форматированием в схему UserProfile
response = client.models.generate_content(
    model='gemini-3.5-flash',  # Изменили версию модели
    contents=f"Проанализируй историю просмотров пользователя и выдели предпочтения вместе с рекомендациями:\n\n{user_log_text}",
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=UserProfile,
        temperature=0.1
    ),
)


# Проверка валидности полученных данных через Pydantic
raw_json = json.loads(response.text)
user_profile = UserProfile(**raw_json)

# Вывод форматированного результата
print("--- СТРУКТУРИРОВАННЫЙ ПРОФИЛЬ ПОЛЬЗОВАТЕЛЯ ---")
print(json.dumps(user_profile.model_dump(), indent=2, ensure_ascii=False))

--- СТРУКТУРИРОВАННЫЙ ПРОФИЛЬ ПОЛЬЗОВАТЕЛЯ ---
{
  "user_id": "USR-9041",
  "watch_history": [
    {
      "title": "Интерстеллар",
      "genre": "Фантастика",
      "rating": 9.5,
      "watch_time_minutes": 169,
      "completion_rate": 100.0
    },
    {
      "title": "Мальчишник в Вегасе",
      "genre": "Комедия",
      "rating": null,
      "watch_time_minutes": 20,
      "completion_rate": 20.0
    },
    {
      "title": "Начало",
      "genre": "Фантастика",
      "rating": 9.0,
      "watch_time_minutes": 148,
      "completion_rate": 100.0
    }
  ],
  "top_genres": [
    "Фантастика"
  ],
  "engagement_level": "Высокий",
  "recommended_movies": [
    "Довод",
    "Бегущий по лезвию 2049",
    "Матрица",
    "Престиж",
    "Прибытие"
  ]
}


In [ ]:
import time

print("Отправка запроса к Gemini...")

for attempt in range(3):
    try:
        response = client.models.generate_content(
            model='gemini-3.5-flash',
            contents=f"Проанализируй историю просмотров пользователя и выдели предпочтения вместе с рекомендациями:\n\n{user_log_text}",
            config=types.GenerateContentConfig(
                response_mime_type="application/json",
                response_schema=UserProfile,
                temperature=0.1
            ),
        )
        print("Успешно получено!")
        break
    except Exception as e:
        print(f"Попытка {attempt + 1} не удалась из-за перегрузки: {e}")
        if attempt < 2:
            print("Ждем 5 секунд перед повтором...")
            time.sleep(5)
        else:
            raise

# Проверка валидности полученных данных через Pydantic
raw_json = json.loads(response.text)
user_profile = UserProfile(**raw_json)

# Вывод форматированного результата
print("\n--- СТРУКТУРИРОВАННЫЙ ПРОФИЛЬ ПОЛЬЗОВАТЕЛЯ ---")
print(json.dumps(user_profile.model_dump(), indent=2, ensure_ascii=False))

Отправка запроса к Gemini...
Успешно получено!

--- СТРУКТУРИРОВАННЫЙ ПРОФИЛЬ ПОЛЬЗОВАТЕЛЯ ---
{
  "user_id": "USR-9041",
  "watch_history": [
    {
      "title": "Интерстеллар",
      "genre": "Фантастика",
      "rating": 9.5,
      "watch_time_minutes": 169,
      "completion_rate": 100.0
    },
    {
      "title": "Мальчишник в Вегасе",
      "genre": "Комедия",
      "rating": null,
      "watch_time_minutes": 20,
      "completion_rate": 20.0
    },
    {
      "title": "Начало",
      "genre": "Фантастика",
      "rating": 9.0,
      "watch_time_minutes": 148,
      "completion_rate": 100.0
    }
  ],
  "top_genres": [
    "Фантастика"
  ],
  "engagement_level": "Высокий",
  "recommended_movies": [
    "Престиж",
    "Матрица",
    "Дюна",
    "Бегущий по лезвию 2049"
  ]
}


In [ ]:

# Проверка работы валидатора Pydantic на ошибку
try:
    invalid_movie = WatchedMovie(
        title="Тестовый фильм",
        genre="Драма",
        rating=8.0,
        watch_time_minutes=120,
        completion_rate=150.0  # Ошибка: процент > 100!
    )
except Exception as e:
    print("Успешно сработала проверка валидации:")
    print(e)


Успешно сработала проверка валидации:
1 validation error for WatchedMovie
completion_rate
  Value error, Процент досмотра должен быть в диапазоне от 0 до 100 [type=value_error, input_value=150.0, input_type=float]
    For further information visit https://errors.pydantic.dev/2.14/v/value_error
